# 7회차 · 내 데이터로 모델 학습시키기(LoRA 샌드박스)

오늘은 모델의 **가중치를 직접 바꿈**. 내가 원하는 말투 · 형식을 정하고, 학습 데이터를 만들고, 설정을 바꿔 가며 여러 번 학습시켜 차이를 확인함.
마지막에는 6회차에 만든 내 문서 챗봇에 학습한 Adapter 를 얹어 **내 솔루션**을 완성함.

- 프롬프트로 시키는 것과 학습시키는 것의 차이 : 학습한 모델은 **지시문 없이도** 목표대로 답함
- 학습은 한 번에 1~3분임. 설정을 바꿔 **세 번 이상** 학습해 봄
- 끝낼 때 맨 아래 '결과 묶기'를 실행해 zip 파일을 내려받음

| 단계 | 하는 일 | 필수 |
|---|---|---|
| 0 | 시작 : 6회차 솔루션 불러오기 | 필수 |
| 1 | 학습 목표 정하기 | 필수 |
| 2 | 학습 전 답 확인 | 필수 |
| 3 | 학습 데이터 만들기 | 필수 |
| 4 | 첫 학습 | 필수 |
| 5 | 전후 비교 | 필수 |
| 6 | 설정 실험 : 다시 학습하고 겹쳐 보기 | 필수 |
| 7 | 일부러 망가뜨려 보기 | 선택 |
| 8 | 큰 GPU 써 보기(QLoRA · 큰 배치 · 다른 본체) | 선택 |
| 9 | 네 구성 비교 | 필수 |
| 10 | 내 솔루션 완성 · 저장 | 필수 |
| 11 | 판단 기록 · 결과 묶기 | 필수 |

## 0. 시작
아래 두 셀을 차례로 실행함. 6회차에 저장한 솔루션(모델 · 문서 · 검색 설정)을 그대로 불러옴. 6회차 결과가 없으면 `lab.docs_panel()` 로 문서를 넣고 `lab.quiz_panel()` 로 확인 질문을 적은 뒤 이어 감.

In [ ]:
import sys, pathlib
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "lab.py").exists())
sys.path.insert(0, str(root / "src"))
import lab
lab.start(7)

In [ ]:
lab.load_solution()
lab.gpu()

## 1. 학습 목표 정하기
모델이 **어떻게 답하게 만들 것인지** 정함. 준비된 목표를 고르면 칸이 채워지고, 그대로 쓰거나 고쳐 씀. 직접 새로 적어도 됨.
- **지시문** : 원하는 말투 · 형식 · 길이. 학습 데이터를 만들 때만 쓰고, 학습한 모델에는 주지 않음
- **표지 단어** : 목표대로 답했는지 가려내는 단어나 말끝. 점수 계산에 씀
- **예시** : `질문 => 내가 원하는 답`. 내가 원하는 답의 모습 그대로 3건 이상 적음. 예시의 품질이 학습 결과를 좌우함

In [ ]:
lab.goal_panel()

## 2. 학습 전 답 확인
학습하기 전의 모델에게 지시문 없이 물어봄. 지금은 평범하게 답하는 것이 정상임. 같은 질문을 4단계 뒤에 다시 물어 비교함.

In [ ]:
lab.compare_panel()

## 3. 학습 데이터 만들기
학습에는 `질문 → 원하는 답` 쌍이 수백 건 필요함. 손으로 다 쓸 수 없으므로 지시문과 예시를 받은 **교사 모델**이 질문 수백 건에 목표대로 답하게 하고, 그 답을 학습 데이터로 씀.
- 진행 막대의 '한 번에 몇 건'이 GPU 메모리에 따라 정해짐. 메모리가 클수록 한꺼번에 많이 생성함
- 만들어진 표를 읽어 봄. 답이 목표와 다르면 1단계로 돌아가 지시문 · 예시를 고친 뒤 다시 만듦
- '내 문서 섞는 비율'을 주면 문서를 붙여 답하는 형태도 학습 데이터에 들어감

In [ ]:
lab.data_panel()

## 4. 첫 학습
기본 설정 그대로 '학습 시작'을 누름. Loss 곡선이 실시간으로 그려짐.
- **Loss** : 모델의 답이 학습 데이터의 답과 얼마나 다른지 나타내는 값. 내려가면 학습 데이터를 닮아 가고 있다는 뜻임
- 실선은 학습 Loss, 점선은 학습에 쓰지 않은 검증 데이터의 Loss 임
- 표에서 **학습한 파라미터 비율**과 **Adapter 크기**를 확인함. 본체는 그대로 두고 작은 덧붙임만 학습함

In [ ]:
lab.train_panel()

## 5. 전후 비교
2단계와 같은 질문, 그리고 학습 데이터에 없던 새 질문으로 물어봄. 표지 단어는 색으로 표시됨.
아래 두 번째 셀은 검증 질문 20건으로 **말투 점수**를 재서 막대로 그림.

In [ ]:
lab.compare_panel()

In [ ]:
lab.style_eval()

## 6. 설정 실험
4단계 조절판으로 돌아가 **설정 하나만** 바꿔 다시 학습함(이름은 비워 두면 v2, v3 … 로 붙음). 실행 전에 결과를 예측해 봄.

| 바꿔 볼 것 | 살펴볼 것 |
|---|---|
| rank 2 대 64 | 학습한 파라미터 수 · Adapter 크기 · 말투 점수 |
| 에폭 1 대 6 | Loss 가 어디까지 내려가는가 · 답이 학습 데이터를 그대로 외우는가 |
| 학습 건수 40 대 전부 | 데이터가 적어도 되는가 |
| 붙일 층 q · v 대 전체 | 학습 시간 · 점수 |

두 번 이상 더 학습한 뒤 아래 셀로 Loss 곡선을 겹쳐 그리고 점수를 다시 잼.

In [ ]:
lab.loss_plot()
lab.style_eval()

## 7. 일부러 망가뜨려 보기(선택)
잘못된 설정이 어떤 모습으로 나타나는지 직접 봄. 조절판에서 아래처럼 고르고 학습한 뒤 5단계 조절판으로 답을 읽어 봄.
- **학습률 0.01** : Loss 가 내려가지 않거나 치솟음. 답이 깨지는지 확인함
- **에폭 12 · 학습 건수 20** : Loss 는 0 에 가까워지는데 새 질문에는 엉뚱하게 답함(과적합)
- 1단계에서 예시를 일부러 엉망으로 적고 데이터를 다시 만들면, 모델은 그 엉망인 답을 그대로 배움

## 8. 큰 GPU 써 보기(선택)
- **QLoRA** : 본체 자료형을 4bit 로 바꿔 학습함. 표의 '학습 메모리'와 걸린 시간, 말투 점수를 bf16 과 비교함
- **큰 배치** : 배치를 32 · 64 로 키움. 한 번에 많이 보지만 갱신 횟수가 줄어듦. 점수가 어떻게 되는지 봄
- **다른 본체** : '학습할 본체'를 `Qwen3-8B` 로 바꿔 같은 데이터로 학습함. 그 Adapter 는 그 본체에서만 쓸 수 있으므로, 아래 셀로 본체를 올리고 비교함

In [ ]:
lab.gpu()
# 다른 본체로 학습했으면: lab.load('Qwen/Qwen3-8B') 뒤 lab.compare('질문') 으로 비교하고, 끝나면 lab.use('A.X-4.0-Light') 로 돌아옴

## 9. 네 구성 비교
6회차에 적은 확인 질문 전체를 네 가지 구성으로 풀어 **내용 점수**(기대 단어)와 **말투 점수**(표지 단어)를 함께 봄.
- 학습 전 · 문서 붙임(RAG) · Adapter(LoRA) · 문서 + Adapter
- 내용은 무엇이 올리고, 말투는 무엇이 올리는가. 둘을 같이 쓰면 어떻게 되는가

In [ ]:
lab.four()

## 10. 내 솔루션 완성
가장 마음에 드는 Adapter 이름을 넣어 저장함. 저장한 뒤 대화 창에서 써 봄. 8회차부터는 이 솔루션을 불러 써서 서비스로 만듦.

In [ ]:
lab.runs()

In [ ]:
lab.save_solution(adapter='v1')     # 고른 Adapter 이름으로 바꿈

In [ ]:
lab.chat()

## 11. 판단 기록 · 결과 묶기
어떤 설정의 Adapter 를 골랐고 왜 골랐는지, 무엇이 기대와 달랐는지 적음. 끝으로 결과 묶기를 실행하고 zip 파일을 내려받음.

In [ ]:
lab.note_panel()

In [ ]:
lab.pack()

---
### 더 해 보기
- 조절판 없이 직접 부르기 : `lab.train(r=8, epochs=2)`, `lab.compare("질문", rag=True)`, `lab.style_eval()`
- 5회차의 도메인 과제(JSON 답)를 학습시키려면 : `lab.sh("python src/build_train_data.py --augment 300")` 뒤 `lab.sh("python src/train_lora.py --run r16")`